<a href="https://colab.research.google.com/github/chk-33/large-language-modelloing/blob/main/streamlitapp%20for%20llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import streamlit as st

# Set page configuration
st.set_page_config(
    page_title="Shakespeare Text Generator",
    page_icon="🎭",
    layout="centered"
)

# -----------------------------------------------------------------------------
# 1. MODEL ARCHITECTURE
# (Replace/Update these classes with your actual PyTorch model implementation)
# -----------------------------------------------------------------------------

# Example Hyperparameters matching your notebook setup:
VOCAB_SIZE = 65
D_MODEL = 128
NUM_LAYERS = 2
NUM_HEADS = 2
MAX_SEQ_LEN = 256

class CharacterTransformer(nn.Module):
    """
    Placeholder/Wrapper structure for your Transformer model.
    Ensure this architecture matches the exact model definition used during training.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(MAX_SEQ_LEN, d_model)
        # Add your Attention / MoE / FeedForward layers here
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.pos_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        logits = self.lm_head(x)
        return logits


# -----------------------------------------------------------------------------
# 2. VOCABULARY & MODEL LOADING
# -----------------------------------------------------------------------------

@st.cache_resource
def load_vocab_and_model(checkpoint_path="best_model.pt"):
    """
    Loads character mappings and model state into memory.
    Cached so it only runs once when the app starts.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # TinyShakespeare standard character set mapping
    chars = [
        '\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?',
        'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
        'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
        'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'
    ]

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    model = CharacterTransformer()

    # Load model weights if a checkpoint exists
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint)
        st.sidebar.success("Model weights loaded successfully!")
    except Exception as e:
        st.sidebar.warning(f"Could not load checkpoint directly ({e}). Running model with initialized weights.")

    model.to(device)
    model.eval()
    return model, stoi, itos, device


# -----------------------------------------------------------------------------
# 3. TEXT GENERATION FUNCTION
# -----------------------------------------------------------------------------

def generate_text(model, prompt, stoi, itos, device, max_new_tokens=200, temperature=1.0, top_k=None):
    """
    Generates new tokens autoregressively based on the user prompt.
    """
    model.eval()

    # Encode prompt (fallback to ' ' if characters are unknown)
    idx = torch.tensor([[stoi.get(ch, 0) for ch in prompt]], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Crop context to max_seq_len
        idx_cond = idx[:, -MAX_SEQ_LEN:]

        with torch.no_grad():
            logits = model(idx_cond)
            # Focus only on the last time step
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            # Optional Top-K Filtering
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

    # Decode sequence
    generated_indices = idx[0].tolist()
    return "".join([itos.get(i, "") for i in generated_indices])


# -----------------------------------------------------------------------------
# 4. STREAMLIT UI
# -----------------------------------------------------------------------------

st.title("🎭 Shakespeare Language Model")
st.markdown("Generate custom text using your trained character-level Transformer model.")

# Sidebar Configuration
st.sidebar.header("Generation Settings")

temperature = st.sidebar.slider("Temperature", min_value=0.1, max_value=2.0, value=0.8, step=0.05,
                                help="Higher values increase randomness/creativity, lower values make predictions more deterministic.")

top_k = st.sidebar.slider("Top-K Sampling", min_value=1, max_value=65, value=10, step=1,
                          help="Restricts token sampling to the top K most likely options.")

max_tokens = st.sidebar.slider("Max New Tokens", min_value=50, max_value=1000, value=300, step=50)

# Main UI Prompt Input
prompt_input = st.text_area("Initial Prompt / Seed Text:", value="First Citizen:\nWhat wouldst I beg.", height=120)

if st.button("Generate Text", type="primary"):
    if not prompt_input.strip():
        st.warning("Please enter a starting prompt.")
    else:
        with st.spinner("Generating response..."):
            model, stoi, itos, device = load_vocab_and_model()

            output_text = generate_text(
                model=model,
                prompt=prompt_input,
                stoi=stoi,
                itos=itos,
                device=device,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_k=top_k
            )

            st.subheader("Generated Output:")
            st.code(output_text, language="text")